In [209]:
from itertools import combinations

import pandas as pd

In [210]:
#predict probability of winning
def predict_proba(A_elo, B_elo):
    return 1 / (1 + 10 ** ((B_elo - A_elo) / 400))

In [211]:
elo_teams = pd.read_csv('../../data/final_elo_team.csv')
elo_teams

,TeamID,TeamELO
0,1228,1964.839260
1,1328,1829.561010
2,1106,1144.622636
3,1354,1254.127888
4,1112,1983.402048
...,...,...
744,1478,1160.320470
745,1479,1341.454910
746,1480,1258.946093
747,3480,1369.079959


In [212]:
elo_coaches = pd.read_csv('../../data/final_elo_coach.csv')
elo_coaches

,Season,CoachName,CoachELO,TeamID
0,2025.0,fran_mccaffery,1835.362744,1234.0
1,2025.0,rick_pitino,2097.639651,1385.0
2,2025.0,leonard_hamilton,1785.414102,1199.0
3,2025.0,rick_barnes,2137.876613,1397.0
4,2025.0,kelvin_sampson,2221.077399,1222.0
...,...,...,...,...
359,2025.0,craig_doty,1421.790908,1223.0
360,2025.0,paul_corsaro,1404.101904,1237.0
361,2025.0,cornelius_jackson,1610.949493,1267.0
362,2025.0,ethan_faulkner,1587.545769,1464.0


In [213]:
df = elo_teams.merge(elo_coaches, how='left', on='TeamID')
df

,TeamID,TeamELO,Season,CoachName,CoachELO
0,1228,1964.839260,2025.0,brad_underwood,2019.328587
1,1328,1829.561010,2025.0,porter_moser,1882.173534
2,1106,1144.622636,2025.0,tony_madlock,1396.494887
3,1354,1254.127888,2025.0,erik_martin,1502.988981
4,1112,1983.402048,2025.0,tommy_lloyd,2029.570166
...,...,...,...,...,...
744,1478,1160.320470,2025.0,nate_champion,1295.729496
745,1479,1341.454910,2025.0,gary_manchel,1448.532417
746,1480,1258.946093,2025.0,dave_moore,1342.672013
747,3480,1369.079959,NaN,NaN,NaN


In [214]:
#drop rows if there is no Elo for team or coach and TeamID is lower than 3000
df = df[~((df['TeamID'] < 3000) & (df[['CoachELO']].isna().any(axis=1)))]
df

,TeamID,TeamELO,Season,CoachName,CoachELO
0,1228,1964.839260,2025.0,brad_underwood,2019.328587
1,1328,1829.561010,2025.0,porter_moser,1882.173534
2,1106,1144.622636,2025.0,tony_madlock,1396.494887
3,1354,1254.127888,2025.0,erik_martin,1502.988981
4,1112,1983.402048,2025.0,tommy_lloyd,2029.570166
...,...,...,...,...,...
744,1478,1160.320470,2025.0,nate_champion,1295.729496
745,1479,1341.454910,2025.0,gary_manchel,1448.532417
746,1480,1258.946093,2025.0,dave_moore,1342.672013
747,3480,1369.079959,NaN,NaN,NaN


In [215]:
df = df.drop(columns=['CoachName'])
df = df.sort_values(by=['TeamID'])

In [216]:
only_men_teams = df[df['TeamID'] < 3000]
only_women_teams = df[df['TeamID'] >= 3000]
print(only_women_teams.head())
print(only_men_teams.head())

     TeamID      TeamELO  Season  CoachELO
715    3101  1446.153757     NaN       NaN
517    3102  1451.722953     NaN       NaN
331    3103  1308.361933     NaN       NaN
314    3104  2029.354262     NaN       NaN
628    3105  1327.768099     NaN       NaN
     TeamID      TeamELO  Season     CoachELO
712    1101  1456.306220  2025.0  1555.327427
55     1102  1288.279337  2025.0  1392.758179
220    1103  1701.862618  2025.0  1829.871273
103    1104  2073.372381  2025.0  2134.930218
637    1105  1055.298753  2025.0  1303.680308


# Predictions based on Elo, if both teams have coach Elo do predictions based on coach Elo and team elo and mean of both, otherwise do predictions based on team Elo
firstly do the matchups dataframe

In [217]:
df = pd.DataFrame(only_men_teams)

# Tworzenie wszystkich unikalnych kombinacji drużyn
matches = []
for team1, team2 in combinations(df.itertuples(index=False), 2):
    if team1.TeamID < team2.TeamID:
        matches.append((team1.TeamID, team1.TeamELO, team1.CoachELO,
                        team2.TeamID, team2.TeamELO, team2.CoachELO, team1.Season))
    else:
        matches.append((team2.TeamID, team2.TeamELO, team2.CoachELO,
                        team1.TeamID, team1.TeamELO, team1.CoachELO, team1.Season))

# Tworzenie nowej ramki danych
columns = ['Team1_ID', 'Team1_ELO', 'Team1_CoachELO',
           'Team2_ID', 'Team2_ELO', 'Team2_CoachELO', 'Season']
df_matches = pd.DataFrame(matches, columns=columns)

print(df_matches)

       Team1_ID   Team1_ELO  Team1_CoachELO  Team2_ID    Team2_ELO  \
0          1101  1456.30622     1555.327427      1102  1288.279337   
1          1101  1456.30622     1555.327427      1103  1701.862618   
2          1101  1456.30622     1555.327427      1104  2073.372381   
3          1101  1456.30622     1555.327427      1105  1055.298753   
4          1101  1456.30622     1555.327427      1106  1144.622636   
...         ...         ...             ...       ...          ...   
66061      1477  1099.50187     1292.013185      1479  1341.454910   
66062      1477  1099.50187     1292.013185      1480  1258.946093   
66063      1478  1160.32047     1295.729496      1479  1341.454910   
66064      1478  1160.32047     1295.729496      1480  1258.946093   
66065      1479  1341.45491     1448.532417      1480  1258.946093   

       Team2_CoachELO  Season  
0         1392.758179  2025.0  
1         1829.871273  2025.0  
2         2134.930218  2025.0  
3         1303.680308  2025.0  

In [218]:
#check if Team1_ID is lower than Team2_ID
(df_matches['Team1_ID'] < df_matches['Team2_ID']).all()

True

In [219]:
df_matches['Pred_team'] = df_matches.apply(lambda x: predict_proba(x['Team1_ELO'], x['Team2_ELO']), axis=1)
df_matches['Pred_coach'] = df_matches.apply(lambda x: predict_proba(x['Team1_CoachELO'], x['Team2_CoachELO']), axis=1)
df_matches['Pred'] = (df_matches['Pred_team'] + df_matches['Pred_coach']) / 2
df_matches

,Team1_ID,Team1_ELO,Team1_CoachELO,Team2_ID,Team2_ELO,Team2_CoachELO,Season,Pred_team,Pred_coach,Pred
0,1101,1456.30622,1555.327427,1102,1288.279337,1392.758179,2025.0,0.724569,0.718255,0.721412
1,1101,1456.30622,1555.327427,1103,1701.862618,1829.871273,2025.0,0.195677,0.170739,0.183208
2,1101,1456.30622,1555.327427,1104,2073.372381,2134.930218,2025.0,0.027865,0.034341,0.031103
3,1101,1456.30622,1555.327427,1105,1055.298753,1303.680308,2025.0,0.909569,0.809782,0.859676
4,1101,1456.30622,1555.327427,1106,1144.622636,1396.494887,2025.0,0.857441,0.713882,0.785661
...,...,...,...,...,...,...,...,...,...,...
66061,1477,1099.50187,1292.013185,1479,1341.454910,1448.532417,2025.0,0.198962,0.288846,0.243904
66062,1477,1099.50187,1292.013185,1480,1258.946093,1342.672013,2025.0,0.285399,0.427608,0.356504
66063,1478,1160.32047,1295.729496,1479,1341.454910,1448.532417,2025.0,0.260631,0.293260,0.276945
66064,1478,1160.32047,1295.729496,1480,1258.946093,1342.672013,2025.0,0.361760,0.432852,0.397306


In [220]:
#if pred is Nan then Pred=Pred_team
df_matches['Pred'] = df_matches['Pred'].fillna(df_matches['Pred_team'])
df_matches

,Team1_ID,Team1_ELO,Team1_CoachELO,Team2_ID,Team2_ELO,Team2_CoachELO,Season,Pred_team,Pred_coach,Pred
0,1101,1456.30622,1555.327427,1102,1288.279337,1392.758179,2025.0,0.724569,0.718255,0.721412
1,1101,1456.30622,1555.327427,1103,1701.862618,1829.871273,2025.0,0.195677,0.170739,0.183208
2,1101,1456.30622,1555.327427,1104,2073.372381,2134.930218,2025.0,0.027865,0.034341,0.031103
3,1101,1456.30622,1555.327427,1105,1055.298753,1303.680308,2025.0,0.909569,0.809782,0.859676
4,1101,1456.30622,1555.327427,1106,1144.622636,1396.494887,2025.0,0.857441,0.713882,0.785661
...,...,...,...,...,...,...,...,...,...,...
66061,1477,1099.50187,1292.013185,1479,1341.454910,1448.532417,2025.0,0.198962,0.288846,0.243904
66062,1477,1099.50187,1292.013185,1480,1258.946093,1342.672013,2025.0,0.285399,0.427608,0.356504
66063,1478,1160.32047,1295.729496,1479,1341.454910,1448.532417,2025.0,0.260631,0.293260,0.276945
66064,1478,1160.32047,1295.729496,1480,1258.946093,1342.672013,2025.0,0.361760,0.432852,0.397306


In [221]:
#submission df is column with ID = 2025_{Team1_ID}_{Team2_ID} and column with Pred = Pred
df_matches['ID'] = '2025_' + df_matches['Team1_ID'].astype(str) + '_' + df_matches['Team2_ID'].astype(str)
submission = df_matches[['ID', 'Pred']]
submission.index = submission['ID']
submission = submission.drop(columns=['ID'])
submission

,Pred
ID,
2025_1101_1102,0.721412
2025_1101_1103,0.183208
2025_1101_1104,0.031103
2025_1101_1105,0.859676
2025_1101_1106,0.785661
...,...
2025_1477_1479,0.243904
2025_1477_1480,0.356504
2025_1478_1479,0.276945


In [222]:
df = pd.DataFrame(only_women_teams)

# Tworzenie wszystkich unikalnych kombinacji drużyn
matches = []
for team1, team2 in combinations(df.itertuples(index=False), 2):
    if team1.TeamID < team2.TeamID:
        matches.append((team1.TeamID, team1.TeamELO, team1.CoachELO,
                        team2.TeamID, team2.TeamELO, team2.CoachELO, team1.Season))
    else:
        matches.append((team2.TeamID, team2.TeamELO, team2.CoachELO,
                        team1.TeamID, team1.TeamELO, team1.CoachELO, team1.Season))

# Tworzenie nowej ramki danych
columns = ['Team1_ID', 'Team1_ELO', 'Team1_CoachELO',
           'Team2_ID', 'Team2_ELO', 'Team2_CoachELO', 'Season']
df_matches = pd.DataFrame(matches, columns=columns)

print(df_matches)

       Team1_ID    Team1_ELO  Team1_CoachELO  Team2_ID    Team2_ELO  \
0          3101  1446.153757             NaN      3102  1451.722953   
1          3101  1446.153757             NaN      3103  1308.361933   
2          3101  1446.153757             NaN      3104  2029.354262   
3          3101  1446.153757             NaN      3105  1327.768099   
4          3101  1446.153757             NaN      3106   974.944784   
...         ...          ...             ...       ...          ...   
67891      3477  1126.698632             NaN      3479  1213.173465   
67892      3477  1126.698632             NaN      3480  1369.079959   
67893      3478  1190.155307             NaN      3479  1213.173465   
67894      3478  1190.155307             NaN      3480  1369.079959   
67895      3479  1213.173465             NaN      3480  1369.079959   

       Team2_CoachELO  Season  
0                 NaN     NaN  
1                 NaN     NaN  
2                 NaN     NaN  
3                 N

In [223]:
df_matches=df_matches.drop(columns=['Season', 'Team1_CoachELO', 'Team2_CoachELO'])
df_matches

,Team1_ID,Team1_ELO,Team2_ID,Team2_ELO
0,3101,1446.153757,3102,1451.722953
1,3101,1446.153757,3103,1308.361933
2,3101,1446.153757,3104,2029.354262
3,3101,1446.153757,3105,1327.768099
4,3101,1446.153757,3106,974.944784
...,...,...,...,...
67891,3477,1126.698632,3479,1213.173465
67892,3477,1126.698632,3480,1369.079959
67893,3478,1190.155307,3479,1213.173465
67894,3478,1190.155307,3480,1369.079959


In [224]:
df_matches['Pred'] = df_matches.apply(lambda x: predict_proba(x['Team1_ELO'], x['Team2_ELO']), axis=1)
df_matches

,Team1_ID,Team1_ELO,Team2_ID,Team2_ELO,Pred
0,3101,1446.153757,3102,1451.722953,0.491986
1,3101,1446.153757,3103,1308.361933,0.688517
2,3101,1446.153757,3104,2029.354262,0.033661
3,3101,1446.153757,3105,1327.768099,0.664070
4,3101,1446.153757,3106,974.944784,0.937760
...,...,...,...,...,...
67891,3477,1126.698632,3479,1213.173465,0.378060
67892,3477,1126.698632,3480,1369.079959,0.198570
67893,3478,1190.155307,3479,1213.173465,0.466923
67894,3478,1190.155307,3480,1369.079959,0.263089


In [225]:
df_matches['ID'] = '2025_' + df_matches['Team1_ID'].astype(str) + '_' + df_matches['Team2_ID'].astype(str)
submission_women = df_matches[['ID', 'Pred']]
submission_women.index = submission_women['ID']
submission_women= submission_women.drop(columns=['ID'])
submission_women

,Pred
ID,
2025_3101_3102,0.491986
2025_3101_3103,0.688517
2025_3101_3104,0.033661
2025_3101_3105,0.664070
2025_3101_3106,0.937760
...,...
2025_3477_3479,0.378060
2025_3477_3480,0.198570
2025_3478_3479,0.466923


In [226]:
#submission is submission and submission_women
submission = pd.concat([submission, submission_women])
submission

,Pred
ID,
2025_1101_1102,0.721412
2025_1101_1103,0.183208
2025_1101_1104,0.031103
2025_1101_1105,0.859676
2025_1101_1106,0.785661
...,...
2025_3477_3479,0.378060
2025_3477_3480,0.198570
2025_3478_3479,0.466923


In [227]:
sample_submission = pd.read_csv('../../data/SampleSubmissionStage2.csv')
sample_submission = sample_submission.drop(columns=['Pred'])
sample_submission

,ID
0,2025_1101_1102
1,2025_1101_1103
2,2025_1101_1104
3,2025_1101_1105
4,2025_1101_1106
...,...
131402,2025_3477_3479
131403,2025_3477_3480
131404,2025_3478_3479
131405,2025_3478_3480


In [228]:
submission = sample_submission.merge(submission, how='left', on='ID')
submission.index = submission['ID']
submission = submission.drop(columns=['ID'])
submission

,Pred
ID,
2025_1101_1102,0.721412
2025_1101_1103,0.183208
2025_1101_1104,0.031103
2025_1101_1105,0.859676
2025_1101_1106,0.785661
...,...
2025_3477_3479,0.378060
2025_3477_3480,0.198570
2025_3478_3479,0.466923


In [229]:
submission.to_csv('../../data/submissions/only_elo_submission.csv')